# 2 · Data preparation
`2_planet_sandbox_Para_BRA_sentinelhub.ipynb` · **Workflow:** 1 · Imagery discovery → 2 · Data preparation → 3 · NDVI analysis

Prepare consistent annual imagery for the Pará AOI.

- Request or load imagery
- Retrieve the relevant bands
- Apply preprocessing
- Create annual composites
- Save the analysis-ready raster outputs

Keeping data retrieval and preprocessing separate ensures that the analysis notebook receives standardized inputs.

**Why Sentinel Hub and not `api.planet.com`:** the 2021-2023 sandbox collection (`PS_ARD_SR_DAILY`, analysis-ready PlanetScope surface reflectance) is hosted on Sentinel Hub.
A Planet API key returns 404 there, so this notebook uses an **OAuth client** (Planet account → *OAuth Clients* → *Create New*; copy the secret immediately, it is shown once).
The data are licensed CC-BY-NC, cover 2021-01-01 to 2023-12-29, and every download uses Processing Units from the account.

**Needs:** `pip install requests pandas numpy rasterio` · **Outputs:** `all_images_sentinel.csv`, `planet_downloads/ARPS_Para_<date>.tif`, `composites/composite_<period>.tif`

## 2.1 Setup

In [ ]:
import time
import warnings
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display
from rasterio.windows import Window

## 2.2 Settings
Everything that defines *what* and *where* is set here once; later cells only use these names.

- `BBOX`: the AOI as `[west, south, east, north]` in EPSG:4326.
- `START_DATE` / `END_DATE`: deliberately wider than the collection, so the search returns exactly what exists.
- `BANDS`: the four surface-reflectance bands, in this order in every file (red = band 3, NIR = band 4).
- `WIDTH` / `HEIGHT`: 2400 px (Process API maximum 2500) over about 24 km ≈ 10 m per pixel; the native resolution is 3 m.
- `STEP`: download every 7th available date to limit Processing Units and time.
- `BASE_URL`: EU deployment. If the collection is not found, try `https://services-uswest2.sentinel-hub.com`.
- Folders are relative to the notebook, so the project can be moved.

In [44]:
COLLECTION_ID = "3f605f75-86c4-411a-b4ae-01c896f0e54e"
COLLECTION = f"byoc-{COLLECTION_ID}"

BBOX = [-52.4825, -6.8735, -52.26725, -6.65775]
BBOX_CRS = "http://www.opengis.net/def/crs/EPSG/0/4326"

START_DATE = "2013-01-01T00:00:00Z"
END_DATE = "2026-01-31T23:59:59Z"

BANDS = ["blue", "green", "red", "nir"]
WIDTH = HEIGHT = 2400
STEP = 7

BASE_URL = "https://services.sentinel-hub.com"

OUTPUT_DIR = Path.cwd() / "planet_downloads"
COMPOSITE_DIR = Path.cwd() / "composites"

print("Collection:", COLLECTION)
print("Date range:", START_DATE, "to", END_DATE)

Collection: byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e
Date range: 2013-01-01T00:00:00Z to 2026-01-31T23:59:59Z


## 2.3 Authenticate
### 2.3.1 Credentials
Typed into hidden prompts and never printed. Empty values stop the run here rather than as a later HTTP 400.

In [45]:
CLIENT_ID = getpass("Planet OAuth client ID: ").strip()
CLIENT_SECRET = getpass("Planet OAuth client secret: ").strip()

if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Client ID and secret are both required.")

print("Credentials loaded.")

Credentials loaded.


### 2.3.2 Access token
The client ID and secret are exchanged for a short-lived bearer token. It is cached and renewed 60 s before it expires, so a long download cannot fail halfway.
Requesting a token now shows a wrong secret immediately.

In [46]:
_token = {"value": None, "expires_at": 0.0}


def get_token():
    """Return a valid bearer token, fetching a new one only when needed."""
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    r = requests.post(
        f"{BASE_URL}/auth/realms/main/protocol/openid-connect/token",
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
        timeout=60,
    )
    if not r.ok:
        print("Token request failed:", r.status_code)
        print(r.text[:1000])
        r.raise_for_status()

    data = r.json()
    _token["value"] = data["access_token"]
    _token["expires_at"] = time.time() + int(data.get("expires_in", 3600))
    return _token["value"]


def auth_headers(**extra):
    return {"Authorization": f"Bearer {get_token()}", **extra}


get_token()
print("Authentication succeeded.")

Authentication succeeded.


## 2.4 Search the catalog
### 2.4.1 Find every date with imagery over the AOI
The Catalog API lists scenes without returning pixels, so it costs no Processing Units.
The two documented catalog paths are aliases; the second is tried only on a 404. Results come in pages of 100 (followed through `context.next`, at most 20 pages).

In [47]:
def catalog_search(body):
    r = None
    for path in ("/api/v1/catalog/1.0.0/search", "/catalog/v1/search"):
        r = requests.post(BASE_URL + path, headers=auth_headers(**{"Content-Type": "application/json"}),
                          json=body, timeout=120)
        if r.status_code != 404:
            break
    return r


body = {"collections": [COLLECTION], "bbox": BBOX, "datetime": f"{START_DATE}/{END_DATE}", "limit": 100}
features = []

for page in range(20):
    r = catalog_search(body)
    if not r.ok:
        print("HTTP status:", r.status_code)
        print(r.text[:2000])
        if r.status_code == 401:
            print("-> Credentials rejected. Check client ID/secret.")
        elif r.status_code == 403:
            print("-> Account may lack access/Monitoring Credits for this collection.")
        elif r.status_code == 404:
            print("-> Try the other deployment (BASE_URL in section 2.2).")
        r.raise_for_status()

    result = r.json()
    features.extend(result.get("features", []))
    next_page = result.get("context", {}).get("next")
    if not next_page:
        break
    body["next"] = next_page

print("Returned features:", len(features))
if not features:
    print("No scenes found. The collection covers 2021-01-01 to 2023-12-29; check the dates and that BBOX is inside the sandbox area.")

Returned features: 677


### 2.4.2 Summarise and save the catalog
One row per scene with its timestamp, any cloud-related property (matched by the word "cloud", because the exact field name is not known in advance) and the date (`YYYY-MM-DD`), since downloads are requested per day.
Saved as `all_images_sentinel.csv`: everything that is *available*, whether or not it is downloaded.

In [ ]:
if not features:
    raise RuntimeError("No scenes found. See the message above.")

rows = []
for f in features:
    props = f.get("properties", {})
    row = {"id": f.get("id"), "datetime": props.get("datetime")}
    row.update({k: v for k, v in props.items() if "cloud" in k.lower()})
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df["date"] = results_df["datetime"].str[:10]
results_df.to_csv("all_images_sentinel.csv", index=False)

print(f"{len(results_df)} images on {results_df['date'].nunique()} dates, {results_df['date'].min()} to {results_df['date'].max()}")
display(results_df.head())

## 2.5 Download the bands
One GeoTIFF per selected date, saved as `ARPS_Para_<date>.tif` (analysis-ready PlanetScope, Pará).

- **Bands and preprocessing:** the evalscript returns `BANDS` plus `dataMask` as 32-bit floats (so reflectance keeps its decimals). `dataMask` is the last band (1 = valid pixel), so each file has 5 bands.
- **One request per day** with `mosaickingOrder: mostRecent` (if several scenes overlap, the most recent is on top), so every file is a clean snapshot of one date.
- **Dates:** every `STEP`-th date plus the last one, so the series always spans the full period.
- **Resumable:** existing files are skipped. The loop stops on HTTP 401/403/429 (credentials, permission or quota cannot be fixed by retrying) or any other error, and pauses 1 s between requests.

In [ ]:
def build_evalscript(bands):
    names = list(bands) + ["dataMask"]
    inputs = ", ".join(f'"{b}"' for b in names)
    outputs = ", ".join(f"sample.{b}" for b in names)
    return f"""//VERSION=3
function setup() {{
  return {{
    input: [{{ bands: [{inputs}] }}],
    output: {{ bands: {len(names)}, sampleType: "FLOAT32" }}
  }};
}}
function evaluatePixel(sample) {{
  return [{outputs}];
}}
"""


def process_request(date):
    payload = {
        "input": {
            "bounds": {"bbox": BBOX, "properties": {"crs": BBOX_CRS}},
            "data": [{
                "type": COLLECTION,
                "dataFilter": {
                    "timeRange": {"from": f"{date}T00:00:00Z", "to": f"{date}T23:59:59Z"},
                    "mosaickingOrder": "mostRecent",
                },
            }],
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": build_evalscript(BANDS),
    }
    headers = auth_headers(**{"Content-Type": "application/json", "Accept": "image/tiff"})
    return requests.post(f"{BASE_URL}/api/v1/process", headers=headers, json=payload, timeout=600)


all_dates = sorted(results_df["date"].astype(str).unique())
picked = all_dates[::STEP]
if all_dates[-1] not in picked:
    picked.append(all_dates[-1])

OUTPUT_DIR.mkdir(exist_ok=True)
print(f"{len(all_dates)} dates in the catalog, {len(picked)} selected (every {STEP}th + the last).")

downloaded, failed = [], []
for date in picked:
    path = OUTPUT_DIR / f"ARPS_Para_{date}.tif"
    if path.exists() and path.stat().st_size > 0:
        print(date, "- already downloaded, skipping")
        downloaded.append(path)
        continue

    try:
        r = process_request(date)
    except Exception as error:
        failed.append(date)
        print(date, "- ERROR:", repr(error))
        break

    if r.ok:
        path.write_bytes(r.content)
        downloaded.append(path)
        print(date, "- saved", f"{path.stat().st_size:,}", "bytes")
    else:
        failed.append(date)
        print(date, "- FAILED", r.status_code, r.text[:500])
        if r.status_code in (401, 403, 429):
            print("Stopping: authentication, permission or rate-limit error.")
            break
    time.sleep(1)

print(f"\nDone. Downloaded: {len(downloaded)}, failed: {len(failed)}. Folder: {OUTPUT_DIR.name}/")
if failed:
    print("Failed dates:", failed)

## 2.6 Create annual composites
Combines the per-date files into one image per period: `composite_2021.tif`, `composite_2022.tif`, `composite_2023.tif` and `composite_All_years.tif` (4 bands: blue, green, red, NIR; float32; NaN where no date has data).
This is the input of notebook 3.

- **Per-pixel median** over the dates: clouds, shadows and haze are extreme values on a single date, so the median ignores them without a cloud mask. The cost is that a composite mixes a whole year, so seasonal differences are averaged in.
- **Only valid pixels count** (`dataMask` > 0), and dates with less than `MIN_COVERAGE` valid area are left out, because they would add almost no information.
- **Processed in strips of `ROWS` rows**, so the full stack of dates never has to fit in memory.
- **Existing composites are never overwritten.** Delete a file to rebuild it. Files are written under a temporary name first, so an interrupted run leaves no half-written composite.

> The code that originally produced the composites in this project was not saved. This cell is a reconstruction (a median, which a comparison with the downloaded files supports) and has only been tested on small synthetic rasters.

In [ ]:
MIN_COVERAGE = 0.10
ROWS = 32


def year_of(path):
    return path.stem.split("_")[-1][:4]


def coverage(path):
    with rasterio.open(path) as src:
        return (src.read(src.count) > 0).mean()


def median_composite(paths, out_path):
    with rasterio.open(paths[0]) as ref:
        profile, height, width = ref.profile, ref.height, ref.width
    for key in ("tiled", "blockxsize", "blockysize"):
        profile.pop(key, None)
    profile.update(count=len(BANDS), dtype="float32", nodata=np.nan, compress="deflate")

    tmp = out_path.with_name(out_path.stem + ".tmp.tif")
    with rasterio.open(tmp, "w", **profile) as dst:
        dst.descriptions = tuple(BANDS)
        for top in range(0, height, ROWS):
            window = Window(0, top, width, min(ROWS, height - top))
            stack = np.full((len(paths), len(BANDS), window.height, width), np.nan, dtype="float32")
            for i, path in enumerate(paths):
                with rasterio.open(path) as src:
                    data = src.read(window=window)
                stack[i] = np.where(data[len(BANDS)] > 0, data[:len(BANDS)], np.nan)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                dst.write(np.nanmedian(stack, axis=0), window=window)
    tmp.replace(out_path)


files = sorted(OUTPUT_DIR.glob("ARPS_Para_*.tif"))
usable = [p for p in files if coverage(p) >= MIN_COVERAGE]
print(f"{len(usable)} of {len(files)} downloaded dates have at least {MIN_COVERAGE:.0%} valid coverage.")

COMPOSITE_DIR.mkdir(exist_ok=True)
periods = {f"composite_{y}.tif": [p for p in usable if year_of(p) == y] for y in ("2021", "2022", "2023")}
periods["composite_All_years.tif"] = usable

for name, paths in periods.items():
    out_path = COMPOSITE_DIR / name
    if out_path.exists():
        print(name, "- exists, skipping (delete it to rebuild)")
    elif not paths:
        print(name, "- no usable images")
    else:
        print(name, f"- median of {len(paths)} images")
        median_composite(paths, out_path)